# WEG Immanuelkirchstraße 26 — Master Data Analysis

**Property:** WEG Immanuelkirchstraße 26, 10405 Berlin  
**Dataset:** 3 buildings · 52 units · 35 owners · 26 tenants · 16 service providers

---
## Entity Relationship

```
LIEGENSCHAFT (1)
  └── GEBÄUDE / Buildings (3)
        └── EINHEITEN / Units (52)
              ├── EIGENTUEMER / Owners (35)  [via eigentuemer.einheit_ids]
              └── MIETER / Tenants (26)       [via mieter.einheit_id + mieter.eigentuemer_id]

DIENSTLEISTER / Service Providers (16)  [linked by service category, not FK]
```

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
COLORS = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#7c3aed', '#0891b2']

einheiten     = pd.read_csv('einheiten.csv')
eigentuemer   = pd.read_csv('eigentuemer.csv')
mieter        = pd.read_csv('mieter.csv')
dienstleister = pd.read_csv('dienstleister.csv')

with open('stammdaten.json') as f:
    stamm = json.load(f)
lie      = stamm['liegenschaft']
gebaeude = pd.DataFrame(stamm['gebaeude'])

print('Datasets loaded:')
for name, df in [('Einheiten',einheiten),('Eigentümer',eigentuemer),('Mieter',mieter),('Dienstleister',dienstleister),('Gebäude',gebaeude)]:
    print(f'  {name:<18} {len(df):>3} rows  {len(df.columns):>2} cols')

---
## 1. Portfolio Overview

In [ ]:
print(f"Property : {lie['name']}")
print(f"Address  : {lie['strasse']}, {lie['plz']} {lie['ort']}")
print(f"Built    : {lie['baujahr']}   Renovated: {lie['sanierung']}")
print()

bldg = gebaeude[['id','hausnr','einheiten','etagen','fahrstuhl','baujahr']].copy()
bldg.columns = ['ID','Hausnr.','Units','Floors','Elevator','Year Built']
display(bldg.set_index('ID'))

print(f"\nUnit breakdown:")
for t, n in einheiten['typ'].value_counts().items():
    area = einheiten[einheiten.typ==t]['wohnflaeche_qm'].sum()
    print(f"  {t:<14} {n:>3} units  {area:>7.0f} m²")

---
## 2. Unit Analysis — Size, Distribution & Types

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Unit Portfolio Analysis', fontsize=15, fontweight='bold', y=1.01)

# 2a. Units per building by type
ax = axes[0, 0]
unit_per_bldg = einheiten.groupby(['haus_id','typ']).size().unstack(fill_value=0)
unit_per_bldg.plot(kind='bar', ax=ax, color=COLORS[:3], edgecolor='white')
ax.set_title('Units per Building by Type')
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Type')

# 2b. Unit type share
ax = axes[0, 1]
type_counts = einheiten['typ'].value_counts()
ax.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
       colors=COLORS[:3], startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Unit Type Distribution')

# 2c. Residential area histogram
ax = axes[1, 0]
wohnungen = einheiten[einheiten.typ=='Wohnung']['wohnflaeche_qm']
ax.hist(wohnungen, bins=12, color=COLORS[0], edgecolor='white')
ax.axvline(wohnungen.mean(), color=COLORS[2], linestyle='--', linewidth=2, label=f'Mean: {wohnungen.mean():.0f} m²')
ax.axvline(wohnungen.median(), color=COLORS[3], linestyle='--', linewidth=2, label=f'Median: {wohnungen.median():.0f} m²')
ax.set_title('Residential Area Distribution')
ax.set_xlabel('m²')
ax.set_ylabel('# Units')
ax.legend()

# 2d. MEA per building
ax = axes[1, 1]
mea = einheiten.groupby('haus_id')['miteigentumsanteil'].sum()
bars = ax.bar(mea.index, mea.values, color=COLORS[:3], edgecolor='white')
for bar, v in zip(bars, mea.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, f'{v:,}', ha='center', fontweight='bold')
ax.set_title('Total MEA (Co-ownership Share) per Building')
ax.set_ylabel('MEA Sum')

plt.tight_layout()
plt.show()

display(wohnungen.describe().round(1).to_frame('Wohnfläche (m²)'))

---
## 3. Owner Analysis — Concentration & Profile

In [ ]:
eigentuemer['unit_list']  = eigentuemer['einheit_ids'].str.split(';')
eigentuemer['unit_count'] = eigentuemer['unit_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Owner Portfolio Analysis', fontsize=15, fontweight='bold')

# 3a. Units per owner
ax = axes[0]
dist = eigentuemer['unit_count'].value_counts().sort_index()
ax.bar(dist.index.astype(str), dist.values, color=COLORS[0], edgecolor='white')
ax.set_title('Units Owned per Owner')
ax.set_xlabel('# Units Owned')
ax.set_ylabel('# Owners')

# 3b. Owner type
ax = axes[1]
owner_type = eigentuemer['anrede'].map({'Herr':'Individual (M)','Frau':'Individual (F)','Firma':'Company'})
tc = owner_type.value_counts()
ax.pie(tc, labels=tc.index, autopct='%1.1f%%', colors=COLORS[:3], startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Owner Type')

# 3c. Language
ax = axes[2]
lang = eigentuemer['sprache'].value_counts()
bars = ax.bar(lang.index.map({'de':'German','en':'English'}), lang.values,
              color=[COLORS[0], COLORS[2]], edgecolor='white')
for bar, v in zip(bars, lang.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v), ha='center', fontweight='bold')
ax.set_title('Owner Language Preference')
ax.set_ylabel('# Owners')

plt.tight_layout()
plt.show()

print(f"Owners with > 1 unit: {(eigentuemer.unit_count > 1).sum()}")
print(f"Board members (Beirat): {eigentuemer.beirat.sum()}")
print(f"SEV mandate: {eigentuemer.sev_mandat.sum()}")
print(f"Self-occupying: {eigentuemer.selbstnutzer.sum()}")
print()
top = eigentuemer[eigentuemer.unit_count > 1][['id','vorname','nachname','firma','unit_count']]\
    .sort_values('unit_count', ascending=False).head(10).copy()
top['name'] = top.apply(lambda r: r['firma'] if pd.notna(r['firma']) else f"{r['vorname']} {r['nachname']}", axis=1)
display(top[['id','name','unit_count']].set_index('id'))

---
## 4. Tenant & Rental Analysis

In [ ]:
mieter['mietbeginn'] = pd.to_datetime(mieter['mietbeginn'])
mieter['mietende']   = pd.to_datetime(mieter['mietende'])
today = pd.Timestamp('2026-04-25')
mieter['active']       = mieter['mietende'].isna() | (mieter['mietende'] > today)
mieter['tenure_years'] = (today - mieter['mietbeginn']).dt.days / 365
mieter['total_rent']   = mieter['kaltmiete'] + mieter['nk_vorauszahlung']
active = mieter[mieter['active']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Tenant & Rental Analysis', fontsize=15, fontweight='bold', y=1.01)

# 4a. Cold rent distribution
ax = axes[0, 0]
ax.hist(active['kaltmiete'], bins=10, color=COLORS[1], edgecolor='white')
ax.axvline(active['kaltmiete'].mean(), color=COLORS[2], linestyle='--', linewidth=2,
           label=f"Mean: €{active['kaltmiete'].mean():.0f}")
ax.set_title('Cold Rent (Kaltmiete) Distribution')
ax.set_xlabel('€ / month')
ax.set_ylabel('# Tenants')
ax.legend()

# 4b. Revenue composition
ax = axes[0, 1]
total_kalt = active['kaltmiete'].sum()
total_nk   = active['nk_vorauszahlung'].sum()
ax.pie([total_kalt, total_nk],
       labels=[f'Kaltmiete\n€{total_kalt:,.0f}/mo', f'NK Advance\n€{total_nk:,.0f}/mo'],
       autopct='%1.1f%%', colors=[COLORS[1], COLORS[3]], startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title(f'Monthly Revenue\nTotal €{total_kalt+total_nk:,.0f}/mo')

# 4c. Lease start by year
ax = axes[1, 0]
by_year = mieter['mietbeginn'].dt.year.value_counts().sort_index()
ax.bar(by_year.index.astype(str), by_year.values, color=COLORS[0], edgecolor='white')
ax.set_title('New Leases by Year (Mietbeginn)')
ax.set_xlabel('Year')
ax.set_ylabel('# Leases Started')
ax.tick_params(axis='x', rotation=45)

# 4d. Tenure distribution
ax = axes[1, 1]
ax.hist(active['tenure_years'], bins=8, color=COLORS[4], edgecolor='white')
ax.axvline(active['tenure_years'].mean(), color=COLORS[2], linestyle='--', linewidth=2,
           label=f"Mean: {active['tenure_years'].mean():.1f} yrs")
ax.set_title('Tenant Tenure (Active Leases)')
ax.set_xlabel('Years')
ax.set_ylabel('# Tenants')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Active leases: {len(active)} / {len(mieter)} total")
print(f"Rent range:    €{active['kaltmiete'].min():.0f} – €{active['kaltmiete'].max():.0f} / month")
print(f"Avg tenure:    {active['tenure_years'].mean():.1f} years")
print(f"Total deposit: €{active['kaution'].sum():,.0f}")

---
## 5. Occupancy Map — Units → Owners → Tenants

In [ ]:
# Build unit-level join table
unit_df = einheiten.copy()

# Attach active tenant
at = active[['einheit_id','id','kaltmiete','total_rent']].rename(columns={'einheit_id':'id','id':'tenant_id'})
unit_df = unit_df.merge(at, on='id', how='left')

# Explode owner→unit mapping
owner_long = eigentuemer[['id','einheit_ids']].assign(
    unit_id=eigentuemer['einheit_ids'].str.split(';')
).explode('unit_id').rename(columns={'unit_id':'id','id':'owner_id'})[['id','owner_id']]
unit_df = unit_df.merge(owner_long, on='id', how='left')

unit_df['occupied'] = unit_df['tenant_id'].notna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Occupancy Overview', fontsize=15, fontweight='bold')

# 5a. Occupancy by building
ax = axes[0]
occ = unit_df.groupby('haus_id')['occupied'].agg(['sum','count'])
occ['vacant'] = occ['count'] - occ['sum']
occ[['sum','vacant']].rename(columns={'sum':'Occupied','vacant':'Not Rented'}).plot(
    kind='bar', ax=ax, color=[COLORS[1], COLORS[2]], edgecolor='white', stacked=True)
ax.set_title('Unit Occupancy by Building')
ax.set_ylabel('# Units')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)

# 5b. Rent vs size scatter
ax = axes[1]
res_rented = unit_df[(unit_df.typ=='Wohnung') & unit_df.occupied].copy()
res_rented['rent_per_m2'] = res_rented['kaltmiete'] / res_rented['wohnflaeche_qm']
sc = ax.scatter(res_rented['wohnflaeche_qm'], res_rented['kaltmiete'],
                c=res_rented['rent_per_m2'], cmap='viridis', s=80, edgecolors='white')
plt.colorbar(sc, ax=ax, label='€/m²')
ax.set_title('Rent vs. Unit Size (Residential, Rented)')
ax.set_xlabel('Area (m²)')
ax.set_ylabel('Kaltmiete (€/mo)')

plt.tight_layout()
plt.show()

total_u = len(unit_df)
occ_n   = unit_df['occupied'].sum()
unrented_res = set(einheiten[einheiten.typ=='Wohnung']['id']) - set(active['einheit_id'])
print(f"Total units rented: {occ_n} / {total_u} ({100*occ_n/total_u:.0f}%)")
print(f"Residential without active tenant: {len(unrented_res)} → {sorted(unrented_res)}")
if len(res_rented):
    print(f"Avg rent/m² (residential): €{res_rented['rent_per_m2'].mean():.2f}")

---
## 6. Service Provider Analysis — Costs & Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Service Provider Analysis', fontsize=15, fontweight='bold')

# 6a. Monthly contract costs
ax = axes[0]
dl_cost = dienstleister.dropna(subset=['vertrag_monatlich']).sort_values('vertrag_monatlich')
bars = ax.barh(dl_cost['branche'], dl_cost['vertrag_monatlich'], color=COLORS[0], edgecolor='white')
for bar, v in zip(bars, dl_cost['vertrag_monatlich']):
    ax.text(v+5, bar.get_y()+bar.get_height()/2, f'€{v:.0f}', va='center', fontsize=9)
ax.set_title('Monthly Contract Cost by Category')
ax.set_xlabel('€ / month')

# 6b. Hourly rates
ax = axes[1]
dl_hr = dienstleister.dropna(subset=['stundensatz'])
bar_colors = [COLORS[1] if r < 80 else COLORS[3] if r < 120 else COLORS[2] for r in dl_hr['stundensatz']]
ax.bar(range(len(dl_hr)), dl_hr['stundensatz'], color=bar_colors, edgecolor='white')
ax.set_xticks(range(len(dl_hr)))
ax.set_xticklabels(dl_hr['branche'], rotation=45, ha='right', fontsize=9)
ax.set_title('Hourly Rates by Category')
ax.set_ylabel('€ / hour')
patches = [mpatches.Patch(color=c, label=l) for c, l in
           zip(COLORS[:3], ['< €80','€80–€120','> €120'])]
ax.legend(handles=patches)

plt.tight_layout()
plt.show()

total_dl = dienstleister['vertrag_monatlich'].sum()
print(f"Total service providers:  {len(dienstleister)}")
print(f"Monthly contract costs:   €{total_dl:,.0f}")
print(f"Annual contract costs:    €{total_dl*12:,.0f}")
print(f"Categories covered: {dienstleister['branche'].nunique()}")

---
## 7. Relational Integrity Check

In [ ]:
valid_units  = set(einheiten['id'])
valid_owners = set(eigentuemer['id'])
valid_bldgs  = set(gebaeude['id'])

all_owner_units = []
for ids in eigentuemer['einheit_ids'].dropna():
    all_owner_units.extend(ids.split(';'))

checks = [
    ('mieter.einheit_id → einheiten.id',         mieter['einheit_id'].isin(valid_units).sum(),    len(mieter)),
    ('mieter.eigentuemer_id → eigentuemer.id',   mieter['eigentuemer_id'].isin(valid_owners).sum(), len(mieter)),
    ('eigentuemer.einheit_ids → einheiten.id',   sum(u in valid_units for u in all_owner_units),   len(all_owner_units)),
    ('einheiten.haus_id → gebaeude.id',           einheiten['haus_id'].isin(valid_bldgs).sum(),     len(einheiten)),
]

print(f'{'Relationship':<47} {'Valid':>6} {'Total':>6} {'Status':>7}')
print('-' * 70)
for name, valid, total in checks:
    status = '✓ OK' if valid == total else f'✗ {total-valid} broken'
    print(f'{name:<47} {valid:>6} {total:>6} {status:>7}')

unowned = valid_units - set(all_owner_units)
unrented_res = set(einheiten[einheiten.typ=='Wohnung']['id']) - set(active['einheit_id'])
print(f'\nCoverage gaps:')
print(f'  Units with no owner record:            {len(unowned)} → {sorted(unowned)}')
print(f'  Residential units without active lease: {len(unrented_res)} → {sorted(unrented_res)}')

---
## 8. Financial Summary

In [ ]:
monthly_income   = active['total_rent'].sum()
monthly_services = dienstleister['vertrag_monatlich'].sum()
monthly_net      = monthly_income - monthly_services

fig, ax = plt.subplots(figsize=(9, 5))
labels = ['Kaltmiete\n(income)', 'NK Advance\n(income)', 'Service\nContracts\n(cost)', 'Est. Net']
values = [active['kaltmiete'].sum(), active['nk_vorauszahlung'].sum(), -monthly_services, monthly_net]
bar_colors = [COLORS[1], COLORS[3], COLORS[2], COLORS[0]]
bars = ax.bar(labels, values, color=bar_colors, edgecolor='white')
for bar, v in zip(bars, values):
    ypos = bar.get_height() if v >= 0 else 0
    ax.text(bar.get_x()+bar.get_width()/2, ypos+30, f'€{abs(v):,.0f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Monthly Financial Summary')
ax.set_ylabel('€ / month')
plt.tight_layout()
plt.show()

print(f"Monthly gross income:      €{monthly_income:>9,.0f}")
print(f"  Kaltmiete:               €{active['kaltmiete'].sum():>9,.0f}")
print(f"  NK Vorauszahlung:        €{active['nk_vorauszahlung'].sum():>9,.0f}")
print(f"Monthly service contracts: €{monthly_services:>9,.0f}")
print(f"{'─'*40}")
print(f"Est. monthly net:          €{monthly_net:>9,.0f}")
print(f"Est. annual net:           €{monthly_net*12:>9,.0f}")
print(f"Total Kaution held:        €{active['kaution'].sum():>9,.0f}")

---
## 9. Data Completeness Audit

In [ ]:
datasets = [
    ('Einheiten',     einheiten),
    ('Eigentümer',    eigentuemer.drop(columns=['unit_list','unit_count'], errors='ignore')),
    ('Mieter',        mieter.drop(columns=['active','tenure_years','total_rent'], errors='ignore')),
    ('Dienstleister', dienstleister),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Column Completeness per Dataset (% non-null)', fontsize=14, fontweight='bold')

for ax, (name, df) in zip(axes, datasets):
    comp = (1 - df.isna().mean()) * 100
    comp_sorted = comp.sort_values()
    bar_colors = [COLORS[1] if v >= 90 else COLORS[3] if v >= 60 else COLORS[2] for v in comp_sorted]
    ax.barh(comp_sorted.index, comp_sorted.values, color=bar_colors, edgecolor='white')
    ax.set_xlim(0, 108)
    ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'{name} ({df.shape[0]} rows)')
    ax.set_xlabel('% complete')
    for i, v in enumerate(comp_sorted.values):
        ax.text(v+1, i, f'{v:.0f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f'{'Dataset':<20} {'Rows':>5} {'Cols':>5} {'Completeness':>13}')
print('-' * 47)
for name, df in datasets:
    pct = 100 * (1 - df.isna().mean().mean())
    print(f'{name:<20} {df.shape[0]:>5} {df.shape[1]:>5} {pct:>12.1f}%')

---
## 10. Key Insights Summary

In [ ]:
print('=' * 58)
print('  WEG IMMANUELKIRCHSTRASSE 26 — KEY INSIGHTS')
print('=' * 58)

wohn = einheiten[einheiten.typ=='Wohnung']
print('\nPORTFOLIO')
print(f'  {len(einheiten)} units across 3 buildings (built 1926–28, renovated 2008)')
print(f'  {len(wohn)} residential ({wohn.wohnflaeche_qm.sum():.0f} m² total)')
print(f'  {(einheiten.typ=="Tiefgarage").sum()} garages · {(einheiten.typ=="Gewerbe").sum()} commercial unit')

print('\nOWNERS')
print(f'  {len(eigentuemer)} owners — {(eigentuemer.unit_count>1).sum()} hold multiple units')
print(f'  {eigentuemer.beirat.sum()} board members · {eigentuemer.sev_mandat.sum()} with SEV mandate')
print(f'  {(eigentuemer.land!="DE").sum()} international owner(s)')

print('\nTENANTS')
print(f'  {len(active)} active leases · avg tenure {active["tenure_years"].mean():.1f} yrs')
print(f'  Rent: €{active["kaltmiete"].min():.0f} – €{active["kaltmiete"].max():.0f} / month')
print(f'  {(active["sprache"]=="en").sum()} English-speaking tenant(s)')

print('\nSERVICE PROVIDERS')
print(f'  {len(dienstleister)} providers · {dienstleister.branche.nunique()} categories')
print(f'  Monthly fixed contracts: €{dienstleister["vertrag_monatlich"].sum():,.0f}')

print('\nFINANCIALS (active leases vs. known fixed costs)')
print(f'  Gross monthly income:  €{monthly_income:,.0f}')
print(f'  Known monthly costs:   €{monthly_services:,.0f}')
print(f'  Estimated monthly net: €{monthly_net:,.0f}')

print('\nDATA QUALITY')
print('  All FK references valid — data is internally consistent')
print(f'  {len(unrented_res)} residential unit(s) without active tenant (owner-occupied/vacant)')
print('=' * 58)